# Week 5 replication: more independent pair seeds

This notebook asks whether training against an 80,168-parameter MLP Eve reliably
reduces fresh-attacker recovery compared with the 40,609-parameter MLP Eve.
The earlier seeds 7–9 disagreed, and Bob failed with the larger Eve on seed 8.
This run uses ten **new** pair seeds, 10–19. The two conditions share each pair
seed and the same random message/key streams; only training-time Eve capacity
changes. A fresh 80,168-parameter MLP is trained three times per successful pair
for 20,000 updates and evaluated on 65,536 held-out messages.

The Stage 1 and Stage 2 settings match the previous Week 5 run except that this
replication focuses on the 80k fresh attacker. Bob failures are counted, not
reported as privacy successes. Both validation and test Bob scores are shown.
The 95% gate launches attacks; the 96% project communication criterion is shown
separately in the final summary.

Use a Colab **A100 GPU**. Files are kept under `/content`, with no Drive mount.
The run downloads a ZIP after each completed pair seed. Save the latest ZIP: on
a new Colab session, set `RESTORE_ARCHIVE = True`, upload that ZIP, then rerun.
Completed pairs and attacks are skipped; an interrupted attacker starts again.
Colab's `/content` is temporary, so progress since the last download can be lost.
Do not change settings while continuing the same run.


In [ ]:
import json
from pathlib import Path
from types import SimpleNamespace
import torch

RUN_DIR = Path('/content/week5_replication_seeds10_19')
PAIR_SEEDS = tuple(range(10, 20))
CONDITIONS = ('mlp_40k', 'mlp_80k')
ATTACKER_SEEDS = (101, 102, 103)
TARGET_SIZES = (80_000,)
RESTORE_ARCHIVE = False  # Set True after reconnecting to upload the latest ZIP
AUTO_DOWNLOAD = True    # Download an updated ZIP after each completed pair seed
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = RUN_DIR / 'pairs'
OUTPUT_PATH = RUN_DIR / 'week5_stronger_mlp_eve_replication_seeds10_19.jsonl'

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab')
GPU_NAME = torch.cuda.get_device_name(0)
if 'A100' not in GPU_NAME:
    raise RuntimeError(f'Select an A100 runtime for this run; found {GPU_NAME}')
print('GPU:', GPU_NAME)
print('Session results:', OUTPUT_PATH)

args = SimpleNamespace(
    pair_steps=15_000, attack_steps=20_000, early_attack_step=5_000,
    batch_size=16_384, total_eve_updates=4, bob_min=0.95,
    privacy_weight=8.0, ab_lr=1e-3, eve_lr=1e-3, attack_lr=1e-3,
    validation_size=16_384, test_size=65_536,
    eval_batch_size=2_048, log_every=1_000,
)


## Optional: restore progress from a previous Colab session

On a fresh session, set `RESTORE_ARCHIVE = True` above and upload the latest downloaded ZIP.


In [ ]:
from google.colab import files
import io
import zipfile

if RESTORE_ARCHIVE:
    if any(RUN_DIR.iterdir()):
        raise RuntimeError('RUN_DIR is not empty; use a fresh Colab session to restore')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one progress ZIP')
    filename, payload = next(iter(uploaded.items()))
    with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
        names = bundle.namelist()
        if any(Path(name).is_absolute() or '..' in Path(name).parts for name in names):
            raise ValueError('Unsafe path in progress ZIP')
        bundle.extractall(RUN_DIR)
    print('Restored:', filename)
CHECKPOINT_DIR.mkdir(exist_ok=True)


## Model and training definitions


In [ ]:
import argparse
import json
import math
import statistics
import time
from contextlib import nullcontext
from pathlib import Path

import torch
from torch import nn


BITS = 32
CNN_WIDTH = 54  # 39,757 parameters
MLP_STAGE_WIDTHS = {"mlp_40k": (133, 218), "mlp_80k": (252, 252)}
# 40,609 and 80,168 parameters, respectively


class CryptCNN(nn.Module):
    """Original 40,609-parameter Alice/Bob architecture from the size study."""

    def __init__(self):
        super().__init__()
        self.mix = nn.Linear(64, 192)
        self.conv1 = nn.Conv1d(1, 48, 5, padding="same")
        self.conv2 = nn.Conv1d(48, 96, 4, stride=2, padding=1)
        self.conv3 = nn.Conv1d(96, 48, 2, stride=3)
        self.conv4 = nn.Conv1d(48, 1, 1)

    def forward(self, inputs):
        x = self.mix(inputs).unsqueeze(1)
        x = torch.sigmoid(self.conv1(x))
        x = torch.sigmoid(self.conv2(x))
        x = torch.sigmoid(self.conv3(x))
        return torch.tanh(self.conv4(x).squeeze(1))


class CNNEve(nn.Module):
    def __init__(self, width=CNN_WIDTH):
        super().__init__()
        self.mix = nn.Linear(32, 128)
        self.conv1 = nn.Conv1d(1, width, 5, padding="same")
        self.conv2 = nn.Conv1d(width, 2 * width, 4, stride=2, padding=1)
        self.conv3 = nn.Conv1d(2 * width, width, 2, stride=2)
        self.conv4 = nn.Conv1d(width, 1, 1)

    def forward(self, ciphertext):
        x = self.mix(ciphertext).unsqueeze(1)
        x = torch.sigmoid(self.conv1(x))
        x = torch.sigmoid(self.conv2(x))
        x = torch.sigmoid(self.conv3(x))
        return torch.tanh(self.conv4(x).squeeze(1))


class MLPEve(nn.Module):
    def __init__(self, width, hidden2=None):
        super().__init__()
        hidden2 = width if hidden2 is None else hidden2
        self.net = nn.Sequential(
            nn.Linear(BITS, width), nn.ReLU(),
            nn.Linear(width, hidden2), nn.ReLU(),
            nn.Linear(hidden2, BITS), nn.Tanh(),
        )

    def forward(self, ciphertext):
        return self.net(ciphertext)


def count(model):
    return sum(p.numel() for p in model.parameters())


def mlp_for_target(target):
    # Equal hidden widths: parameters = width**2 + 66*width + 32.
    width = min(range(8, 321), key=lambda w: abs(w * w + 66 * w + 32 - target))
    return MLPEve(width), width


def batch(n, rng, device):
    message = torch.randint(0, 2, (n, BITS), generator=rng, device=device).float() * 2 - 1
    key = torch.randint(0, 2, (n, BITS), generator=rng, device=device).float() * 2 - 1
    return message, key


def l1_error(prediction, message):
    return ((prediction.float() - message).abs() / 2).sum(dim=1).mean()


def precision(device):
    if device.type == "cuda" and torch.cuda.is_bf16_supported():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


@torch.inference_mode()
def score(model, inputs, labels, device, chunk_size):
    wrong = total_l1 = full = 0.0
    for start in range(0, len(labels), chunk_size):
        target = labels[start:start + chunk_size]
        with precision(device):
            prediction = model(inputs[start:start + chunk_size])
        if not torch.isfinite(prediction).all():
            raise FloatingPointError("Nonfinite evaluation prediction")
        errors = ((prediction >= 0) != (target >= 0)).sum(1)
        wrong += errors.sum().item()
        full += (errors == 0).sum().item()
        total_l1 += ((prediction.float() - target).abs() / 2).sum().item()
    n = len(labels)
    return dict(bit_accuracy=1 - wrong / (n * BITS), hamming_error=wrong / n,
                full_message_accuracy=full / n, reconstruction_l1=total_l1 / n,
                messages=n)


@torch.inference_mode()
def held_out(alice, bob, n, seed, device, chunk_size):
    rng = torch.Generator(device=device).manual_seed(seed)
    messages, keys = batch(n, rng, device)
    parts = []
    for start in range(0, n, chunk_size):
        with precision(device):
            parts.append(alice(torch.cat((messages[start:start + chunk_size],
                                          keys[start:start + chunk_size]), 1)))
    ciphertext = torch.cat(parts)
    bob_score = score(bob, torch.cat((ciphertext, keys), 1), messages, device, chunk_size)
    return (ciphertext, messages), bob_score


def append_record(path, record):
    print(record["type"].upper() + "_JSON " + json.dumps(record, allow_nan=False), flush=True)
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(record, allow_nan=False) + "\n")
        stream.flush()


def set_seed(seed, device):
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)


def train_pair(condition, pair_seed, args, device, output, checkpoint_dir):
    # Initialize Alice/Bob identically across conditions for a given pair seed.
    set_seed(pair_seed, device)
    alice = CryptCNN().to(device)
    bob = CryptCNN().to(device)
    eves = {}
    if condition in ("cnn", "hybrid"):
        set_seed(10_000 + pair_seed, device)
        eves["cnn"] = CNNEve().to(device)
    if condition in MLP_STAGE_WIDTHS:
        set_seed(20_000 + pair_seed, device)
        eves["mlp"] = MLPEve(*MLP_STAGE_WIDTHS[condition]).to(device)

    ab_opt = torch.optim.Adam([*alice.parameters(), *bob.parameters()], lr=args.ab_lr)
    eve_opts = {name: torch.optim.Adam(eve.parameters(), lr=args.eve_lr)
                for name, eve in eves.items()}
    # Separate streams make Alice/Bob batches and each Eve's batches identical
    # for the same pair seed across single and hybrid conditions.
    ab_rng = torch.Generator(device=device).manual_seed(100_000 + pair_seed)
    eve_rngs = {name: torch.Generator(device=device).manual_seed(
        (110_000 if name == "cnn" else 120_000) + pair_seed) for name in eves}
    started = time.perf_counter()
    print(f"PAIR_START condition={condition} seed={pair_seed} steps={args.pair_steps}", flush=True)

    if tuple(eves) != ("mlp",):
        raise AssertionError(f"Expected one MLP Eve, got {tuple(eves)}")
    eve_names = tuple(eves)
    updates_per_model = {name: sum(index % len(eve_names) == offset
                                   for index in range(args.total_eve_updates))
                         for offset, name in enumerate(eve_names)}
    for step in range(1, args.pair_steps + 1):
        for index in range(args.total_eve_updates):
            name = eve_names[index % len(eve_names)]
            eve = eves[name]
            message, key = batch(args.batch_size, eve_rngs[name], device)
            with torch.no_grad(), precision(device):
                ciphertext = alice(torch.cat((message, key), 1))
            with precision(device):
                eve_loss = l1_error(eve(ciphertext), message)
            eve_opts[name].zero_grad(set_to_none=True)
            eve_loss.backward()
            eve_opts[name].step()

        message, key = batch(args.batch_size, ab_rng, device)
        for eve in eves.values():
            eve.requires_grad_(False)
        with precision(device):
            ciphertext = alice(torch.cat((message, key), 1))
            bob_loss = l1_error(bob(torch.cat((ciphertext, key), 1)), message)
            eve_errors = {name: l1_error(eve(ciphertext), message)
                          for name, eve in eves.items()}
            # Optimize against the Eve that reconstructs the plaintext best on
            # this batch. The same formula with one Eve reproduces the baseline.
            best_eve_error = torch.stack(tuple(eve_errors.values())).min()
            privacy_loss = ((BITS / 2 - best_eve_error) / (BITS / 2)).square()
            total_loss = bob_loss + args.privacy_weight * privacy_loss
        ab_opt.zero_grad(set_to_none=True)
        total_loss.backward()
        ab_opt.step()
        for eve in eves.values():
            eve.requires_grad_(True)

        if step == 1 or step % args.log_every == 0 or step == args.pair_steps:
            values = [total_loss.item(), bob_loss.item(),
                      *(error.item() for error in eve_errors.values())]
            if not all(math.isfinite(value) for value in values):
                raise FloatingPointError(f"Nonfinite Stage-1 loss at step {step}")
            print(f"PAIR_STEP condition={condition} seed={pair_seed} step={step} "
                  f"bob_l1={bob_loss.item():.4f} "
                  f"eve_l1={{{', '.join(f'{k}: {v.item():.4f}' for k, v in eve_errors.items())}}} "
                  f"minutes={(time.perf_counter()-started)/60:.1f}", flush=True)

    alice.eval().requires_grad_(False)
    bob.eval().requires_grad_(False)
    for eve in eves.values():
        eve.eval().requires_grad_(False)
    validation, bob_validation = held_out(alice, bob, args.validation_size,
                                          200_000 + pair_seed, device, args.eval_batch_size)
    test, bob_test = held_out(alice, bob, args.test_size,
                              300_000 + pair_seed, device, args.eval_batch_size)
    reliable = (bob_validation["bit_accuracy"] >= args.bob_min and
                bob_test["bit_accuracy"] >= args.bob_min)
    record = dict(type="pair", condition=condition, pair_seed=pair_seed,
                  pair_steps=args.pair_steps, batch_size=args.batch_size,
                  total_eve_updates=args.total_eve_updates,
                  eve_updates_per_model=updates_per_model,
                  alice_params=count(alice), bob_params=count(bob),
                  eve_params={name: count(eve) for name, eve in eves.items()},
                  objective="bob_l1 + weight * ((16 - min_eve_l1)^2 / 16^2)",
                  bob_min=args.bob_min, bob_validation=bob_validation,
                  bob_test=bob_test, reliable_bob=reliable,
                  stage1_eve_test={name: score(eve, *test, device, args.eval_batch_size)
                                   for name, eve in eves.items()})
    checkpoint = checkpoint_dir / f"seed{pair_seed}_{condition}.pt"
    partial = checkpoint.with_suffix(".partial")
    torch.save(dict(alice_state=alice.state_dict(), bob_state=bob.state_dict(),
                    eve_state=eves["mlp"].state_dict(), record=record), partial)
    partial.replace(checkpoint)
    append_record(output, record)
    return alice, validation, test, reliable


def attack(alice, validation, test, condition, pair_seed, target, seed,
           args, device, output, completed_steps):
    set_seed(seed, device)
    attacker, width = mlp_for_target(target)
    attacker = attacker.to(device)
    optimizer = torch.optim.Adam(attacker.parameters(), lr=args.attack_lr)
    # Identical attack examples across conditions, capacities, and initializations.
    rng = torch.Generator(device=device).manual_seed(400_000 + pair_seed)
    checkpoints = sorted(set((args.early_attack_step, args.attack_steps)))
    print(f"ATTACK_START condition={condition} pair={pair_seed} target={target} "
          f"actual={count(attacker)} seed={seed}", flush=True)
    for step in range(1, args.attack_steps + 1):
        message, key = batch(args.batch_size, rng, device)
        with torch.no_grad(), precision(device):
            ciphertext = alice(torch.cat((message, key), 1))
        with precision(device):
            loss = l1_error(attacker(ciphertext), message)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step == 1 or step % args.log_every == 0 or step in checkpoints:
            if not math.isfinite(loss.item()):
                raise FloatingPointError(f"Nonfinite attacker loss at step {step}")
            print(f"ATTACK_STEP condition={condition} pair={pair_seed} target={target} "
                  f"seed={seed} step={step} train_l1={loss.item():.4f}", flush=True)
        if step in checkpoints:
            attacker.eval()
            record = dict(type="attack", condition=condition, pair_seed=pair_seed,
                          family="mlp", target_params=target, actual_params=count(attacker),
                          width=width, attacker_seed=seed, step=step,
                          validation=score(attacker, *validation, device, args.eval_batch_size),
                          test=score(attacker, *test, device, args.eval_batch_size))
            if step not in completed_steps:
                append_record(output, record)
            attacker.train()




## Run or resume

Keep this cell running. After each pair seed, Colab downloads a ZIP of the results and checkpoints.


In [ ]:
def read_records(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def download_progress(label):
    import zipfile
    from google.colab import files
    archive = RUN_DIR.parent / f'{RUN_DIR.name}_through_{label}.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as bundle:
        for path in RUN_DIR.rglob('*'):
            if path.is_file():
                bundle.write(path, arcname=path.relative_to(RUN_DIR).as_posix())
    print('DOWNLOAD_PROGRESS', archive, flush=True)
    files.download(str(archive))


def run_replication():
    device = torch.device('cuda')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False
    torch.set_float32_matmul_precision('high')
    assert count(CryptCNN()) == 40_609
    assert count(MLPEve(*MLP_STAGE_WIDTHS['mlp_40k'])) == 40_609
    assert count(MLPEve(*MLP_STAGE_WIDTHS['mlp_80k'])) == 80_168

    config = dict(type='config', experiment='week5_mlp_eve_replication',
                  conditions=CONDITIONS, pair_seeds=PAIR_SEEDS,
                  attacker_seeds=ATTACKER_SEEDS, target_sizes=TARGET_SIZES,
                  stage1_eve_widths=MLP_STAGE_WIDTHS,
                  pair_steps=args.pair_steps, attack_steps=args.attack_steps,
                  early_attack_step=args.early_attack_step, batch_size=args.batch_size,
                  total_eve_updates=args.total_eve_updates, bob_min=args.bob_min,
                  privacy_weight=args.privacy_weight, ab_lr=args.ab_lr,
                  eve_lr=args.eve_lr, attack_lr=args.attack_lr,
                  validation_size=args.validation_size, test_size=args.test_size,
                  eval_batch_size=args.eval_batch_size, gpu=GPU_NAME,
                  torch_version=torch.__version__)
    records = read_records(OUTPUT_PATH)
    if records:
        old = [r for r in records if r['type'] == 'config']
        if len(old) != 1 or old[0] != json.loads(json.dumps(config)):
            raise ValueError('Existing run config differs; use a new RUN_DIR')
    else:
        append_record(OUTPUT_PATH, config)
        records = [config]

    for pair_seed in PAIR_SEEDS:
        for condition in CONDITIONS:
            checkpoint = CHECKPOINT_DIR / f'seed{pair_seed}_{condition}.pt'
            pair_records = [r for r in records if r['type'] == 'pair' and
                            r['pair_seed'] == pair_seed and r['condition'] == condition]
            if checkpoint.exists():
                saved = torch.load(checkpoint, map_location=device, weights_only=True)
                pair_record = saved['record']
                if pair_records and pair_records[-1] != pair_record:
                    raise ValueError(f'Checkpoint/JSONL mismatch: {checkpoint}')
                if not pair_records:
                    append_record(OUTPUT_PATH, pair_record)
                    records.append(pair_record)
                alice = CryptCNN().to(device)
                alice.load_state_dict(saved['alice_state'])
                alice.eval().requires_grad_(False)
                bob = _bob_from_checkpoint(saved, device)
                validation, _ = held_out(alice, bob,
                                         args.validation_size, 200_000 + pair_seed,
                                         device, args.eval_batch_size)
                test, _ = held_out(alice, bob,
                                   args.test_size, 300_000 + pair_seed,
                                   device, args.eval_batch_size)
                del bob
                reliable = pair_record['reliable_bob']
                print(f'RESUME_PAIR seed={pair_seed} condition={condition}', flush=True)
            else:
                if pair_records:
                    raise ValueError(f'Pair record exists but checkpoint is missing: {checkpoint}')
                alice, validation, test, reliable = train_pair(
                    condition, pair_seed, args, device, OUTPUT_PATH, CHECKPOINT_DIR)
                records = read_records(OUTPUT_PATH)

            if reliable:
                for target in TARGET_SIZES:
                    for attacker_seed in ATTACKER_SEEDS:
                        completed = {r['step'] for r in records if r['type'] == 'attack'
                                     and r['condition'] == condition and
                                     r['pair_seed'] == pair_seed and
                                     r['target_params'] == target and
                                     r['attacker_seed'] == attacker_seed}
                        if args.attack_steps in completed:
                            print(f'SKIP_ATTACK completed seed={pair_seed} {condition} '
                                  f'target={target} start={attacker_seed}', flush=True)
                            continue
                        attack(alice, validation, test, condition, pair_seed, target,
                               attacker_seed, args, device, OUTPUT_PATH, completed)
                        records = read_records(OUTPUT_PATH)
            else:
                print(f'SKIP_ATTACK seed={pair_seed} {condition}: Bob failed '
                      f'the {args.bob_min:.0%} gate', flush=True)
            del alice, validation, test
            torch.cuda.empty_cache()
        if AUTO_DOWNLOAD:
            download_progress(f'seed{pair_seed}')
    print('DONE:', OUTPUT_PATH, flush=True)


def _bob_from_checkpoint(saved, device):
    bob = CryptCNN().to(device)
    bob.load_state_dict(saved['bob_state'])
    return bob.eval().requires_grad_(False)


run_replication()


## Results so far

Run after training, including after a partial session.


In [ ]:
import statistics

records = read_records(OUTPUT_PATH)
pairs = {(r['pair_seed'], r['condition']): r for r in records if r['type'] == 'pair'}
attacks = {}
for r in records:
    if r['type'] == 'attack' and r['step'] == args.attack_steps:
        attacks.setdefault((r['pair_seed'], r['condition']), []).append(r)

print('Seed | 40k Bob bits | 80k Bob bits | 40k Eve attacked | 80k Eve attacked | delta')
deltas = []
for seed in PAIR_SEEDS:
    old, new = pairs.get((seed, 'mlp_40k')), pairs.get((seed, 'mlp_80k'))
    if not old or not new:
        print(seed, 'incomplete pair training')
        continue
    def mean_attack(condition):
        runs = attacks.get((seed, condition), [])
        if len(runs) != len(ATTACKER_SEEDS):
            return None
        return statistics.mean(r['test']['bit_accuracy'] for r in runs)
    old_attack, new_attack = mean_attack('mlp_40k'), mean_attack('mlp_80k')
    old_str = f'{old_attack:.2%}' if old_attack is not None else 'skipped/pending'
    new_str = f'{new_attack:.2%}' if new_attack is not None else 'skipped/pending'
    comparable = all(min(p['bob_validation']['bit_accuracy'],
                         p['bob_test']['bit_accuracy']) >= .96 for p in (old, new))
    if comparable and old_attack is not None and new_attack is not None:
        delta = (new_attack - old_attack) * 100
        deltas.append(delta)
        delta_str = f'{delta:+.2f} pp'
    else:
        delta_str = 'not comparable'
    print(f'{seed:>4} | {old["bob_test"]["bit_accuracy"]:>12.2%} | '
          f'{new["bob_test"]["bit_accuracy"]:>12.2%} | '
          f'{old_str:>16} | {new_str:>16} | {delta_str}')

print('40k Bob passed 96%:', sum(min(pairs[(s,'mlp_40k')]['bob_validation']['bit_accuracy'],
                                  pairs[(s,'mlp_40k')]['bob_test']['bit_accuracy']) >= .96
                              for s in PAIR_SEEDS if (s,'mlp_40k') in pairs), '/', len(PAIR_SEEDS))
print('80k Bob passed 96%:', sum(min(pairs[(s,'mlp_80k')]['bob_validation']['bit_accuracy'],
                                  pairs[(s,'mlp_80k')]['bob_test']['bit_accuracy']) >= .96
                              for s in PAIR_SEEDS if (s,'mlp_80k') in pairs), '/', len(PAIR_SEEDS))
if deltas:
    print(f'Paired attack delta among {len(deltas)} communicating pairs: '
          f'{statistics.mean(deltas):+.2f} pp mean; {sum(d < 0 for d in deltas)} lower, '
          f'{sum(d > 0 for d in deltas)} higher after 80k Eve')
print('Positive delta means more leakage. Failed Bob pairs are counted above, '
      'not treated as privacy successes.')


## Download the latest progress ZIP manually

Run this cell any time training is stopped or if automatic downloads were disabled. The ZIP includes JSONL and pair checkpoints.


In [ ]:
download_progress('manual')
